# Adaptive Control Problem Workbook

このノートブックは、適応制御を **問題形式で考えながら、実際のプロットを見て学ぶ** ための教材です。

進め方は各章で共通です。

1. 問いを読む
2. まず予想を書く
3. セルを実行してプロットを見る
4. 予想と結果の違いを言語化する
5. パラメータを変えて再実験する

> 注意: ここでは直感形成を優先し、証明は最小限にしています。実務・研究では安定性条件、信号条件、ロバスト性を別途きちんと確認します。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True

def step_signal(t):
    return np.ones_like(t)

def rich_signal(t):
    return 0.8*np.sin(0.7*t) + 0.4*np.sin(2.3*t) + 0.3*np.sign(np.sin(0.17*t))

def plot_timeseries(t, series, title, ylabel='value'):
    plt.figure()
    for label, y in series.items():
        plt.plot(t, y, label=label)
    plt.title(title)
    plt.xlabel('time [s]')
    plt.ylabel(ylabel)
    plt.legend()
    plt.show()

## 1. 規範モデルと追従誤差

対象プラントを次の一次系とします。

$$\dot{x} = -a x + b u$$

目標とする規範モデルは次です。

$$\dot{x}_m = -a_m x_m + b_m r$$

### 考える問い

1. プラントの $a, b$ が分からないとき、固定ゲイン制御だけで常に規範モデルへ追従できるでしょうか。
2. ステップ入力だけを使う場合、どのパラメータが見えにくくなるでしょうか。
3. 追従誤差 $e=x-x_m$ が小さくても、推定パラメータが真値に近いとは限るでしょうか。

### 自分の予想

ここに予想を書いてから次へ進んでください。

In [ ]:
def simulate_mrac(T=30.0, dt=0.002, a=1.2, b=0.8, am=2.0, bm=2.0, gamma=3.0, signal='step', normalized=False, projection=False):
    n = int(T/dt) + 1
    t = np.linspace(0, T, n)
    r = step_signal(t) if signal == 'step' else rich_signal(t)
    x = np.zeros(n)
    xm = np.zeros(n)
    kr = np.zeros(n)
    kx = np.zeros(n)
    u = np.zeros(n)
    e = np.zeros(n)
    for k in range(n-1):
        phi = np.array([r[k], x[k]])
        u[k] = kr[k]*r[k] - kx[k]*x[k]
        e[k] = x[k] - xm[k]
        denom = 1.0 + phi @ phi if normalized else 1.0
        dtheta = -gamma * e[k] * phi / denom
        kr_next = kr[k] + dt*dtheta[0]
        kx_next = kx[k] - dt*dtheta[1]
        if projection:
            kr_next = np.clip(kr_next, -8, 8)
            kx_next = np.clip(kx_next, -8, 8)
        kr[k+1] = kr_next
        kx[k+1] = kx_next
        x[k+1] = x[k] + dt*(-a*x[k] + b*u[k])
        xm[k+1] = xm[k] + dt*(-am*xm[k] + bm*r[k])
    u[-1] = kr[-1]*r[-1] - kx[-1]*x[-1]
    e[-1] = x[-1] - xm[-1]
    return {'t': t, 'r': r, 'x': x, 'xm': xm, 'e': e, 'u': u, 'kr': kr, 'kx': kx}

res = simulate_mrac(gamma=3.0, signal='step')
plot_timeseries(res['t'], {'plant x': res['x'], 'model xm': res['xm']}, 'MRAC: plant output vs reference model')
plot_timeseries(res['t'], {'error e': res['e']}, 'tracking error')
plot_timeseries(res['t'], {'kr': res['kr'], 'kx': res['kx']}, 'adaptive gains')

## 2. 学習率 $\gamma$ を変える

### 考える問い

1. $\gamma$ を大きくすると収束は必ず良くなるでしょうか。
2. 速い適応と制御入力の荒さにはどんな関係がありそうでしょうか。
3. 実機で大きすぎる $\gamma$ が危険な理由は何でしょうか。

In [ ]:
gammas = [0.3, 3.0, 20.0]
plt.figure(figsize=(10, 5))
for gamma in gammas:
    r = simulate_mrac(gamma=gamma, signal='rich')
    plt.plot(r['t'], r['e'], label=f'gamma={gamma}')
plt.title('Effect of adaptation gain on tracking error')
plt.xlabel('time [s]')
plt.ylabel('e')
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
for gamma in gammas:
    r = simulate_mrac(gamma=gamma, signal='rich')
    plt.plot(r['t'], r['u'], label=f'gamma={gamma}')
plt.title('Control effort')
plt.xlabel('time [s]')
plt.ylabel('u')
plt.legend()
plt.show()

## 3. 正規化適応則

入力や状態が大きいと、単純な勾配更新は大きく動きすぎることがあります。そこで $1+\phi^T\phi$ で割る正規化を試します。

### 考える問い

1. 正規化は収束を速くするための工夫でしょうか、それとも暴れにくくする工夫でしょうか。
2. 信号が小さいとき、正規化の効果はどれくらい見えるでしょうか。

In [ ]:
plain = simulate_mrac(gamma=20.0, signal='rich', normalized=False)
norm = simulate_mrac(gamma=20.0, signal='rich', normalized=True)
plot_timeseries(plain['t'], {'plain e': plain['e'], 'normalized e': norm['e']}, 'Normalization effect on error')
plot_timeseries(plain['t'], {'plain u': plain['u'], 'normalized u': norm['u']}, 'Normalization effect on control input')
plot_timeseries(plain['t'], {'plain kr': plain['kr'], 'normalized kr': norm['kr']}, 'Normalization effect on kr')

## 4. 射影: パラメータを安全な範囲に閉じ込める

射影は、推定パラメータが事前に決めた範囲から外れないようにする実装上重要な道具です。

### 考える問い

1. 射影を入れると、なぜ制御入力の異常値を抑えやすいのでしょうか。
2. 射影範囲を狭くしすぎると何が起きるでしょうか。

In [ ]:
no_proj = simulate_mrac(a=0.2, b=2.5, gamma=40.0, signal='rich', normalized=False, projection=False)
proj = simulate_mrac(a=0.2, b=2.5, gamma=40.0, signal='rich', normalized=False, projection=True)
plot_timeseries(no_proj['t'], {'no projection u': no_proj['u'], 'projection u': proj['u']}, 'Projection and control effort')
plot_timeseries(no_proj['t'], {'no projection e': no_proj['e'], 'projection e': proj['e']}, 'Projection and tracking error')

## 5. RLSによるオンライン同定

離散時間のARXモデルを考えます。

$$y_{k+1} = \theta_1 y_k + \theta_2 u_k$$

RLSで $\theta$ を逐次推定します。

### 考える問い

1. ステップ入力だけで2つのパラメータを安定して推定できるでしょうか。
2. 忘却係数 $\lambda$ を小さくすると、何に強くなり何に弱くなるでしょうか。
3. 推定が正しくても、制御がうまくいくとは限らない理由は何でしょうか。

In [ ]:
def simulate_rls(T=40.0, dt=0.02, theta_true=(0.96, 0.08), lam=0.995, signal='rich', noise=0.01):
    n = int(T/dt) + 1
    t = np.linspace(0, T, n)
    u = step_signal(t) if signal == 'step' else rich_signal(t)
    y = np.zeros(n)
    th = np.zeros((n, 2))
    theta = np.array([0.2, 0.2], dtype=float)
    P = 100*np.eye(2)
    rng = np.random.default_rng(7)
    for k in range(n-1):
        y[k+1] = theta_true[0]*y[k] + theta_true[1]*u[k] + noise*rng.normal()
        phi = np.array([y[k], u[k]])
        K = P @ phi / (lam + phi @ P @ phi)
        err = y[k+1] - phi @ theta
        theta = theta + K*err
        P = (P - np.outer(K, phi) @ P) / lam
        th[k+1] = theta
    return {'t': t, 'u': u, 'y': y, 'theta': th}

for sig in ['step', 'rich']:
    rr = simulate_rls(signal=sig)
    plt.figure(figsize=(10, 4))
    plt.plot(rr['t'], rr['theta'][:,0], label='theta1 estimate')
    plt.plot(rr['t'], rr['theta'][:,1], label='theta2 estimate')
    plt.axhline(0.96, color='C0', linestyle='--', alpha=0.5, label='theta1 true')
    plt.axhline(0.08, color='C1', linestyle='--', alpha=0.5, label='theta2 true')
    plt.title(f'RLS estimates with {sig} input')
    plt.xlabel('time [s]')
    plt.legend()
    plt.show()

## 6. ミニ課題

次の変更を自分で試してください。

1. `simulate_mrac` の `a`, `b` を変えて、難しいプラントを探す。
2. `gamma` を大きくし、正規化なしで破綻しそうな条件を探す。
3. `signal='step'` と `signal='rich'` で推定パラメータの収束を比較する。
4. RLSの `lam` を `0.98`, `0.995`, `1.0` で比較する。
5. プラントの真値が途中で変わるケースを作り、忘却係数の意味を確認する。

### まとめ問い

- 追従誤差が小さいことと、パラメータ推定が正しいことは同じですか。
- 適応制御で「プロットを必ず見る」べき信号は何ですか。
- 実機適用前に、どんな制限・飽和・安全策を入れるべきですか。